# Build a Speaker-Aware Meeting Intelligence Pipeline with Audio Diarization

Customer-facing teams often have plenty of call recordings and not enough reliable handoff context. A transcript is useful, but a speaker-aware transcript is the unlock: it lets you separate customer pain from seller follow-up, keep evidence next to action items, and route sensitive commitments into a review workflow before they land in a CRM or support system.

This notebook shows how to build a production-style, post-call meeting intelligence pipeline with OpenAI audio diarization. You will:

1. Accept a recorded meeting audio file.
2. Optionally map known speakers using short reference clips.
3. Call `gpt-4o-transcribe-diarize` with `response_format="diarized_json"`.
4. Normalize the speaker-labeled segments into JSON and Markdown.
5. Use structured outputs to extract a meeting brief, decisions, risks, open questions, action items, and a follow-up email draft.
6. Write reviewable artifacts and a local guardrail report.

The default cells run without an API key using a synthetic diarized transcript. Real audio calls are opt-in so the notebook is safe to review top-to-bottom.


## Architecture

![Architecture diagram](images/architecture.svg)

| Layer | Responsibility | Output |
| --- | --- | --- |
| Audio intake | Accept a call recording and optional known-speaker clips. | `meeting.wav`, `Agent=agent.wav` |
| Pipeline runner | Validate inputs, encode references as data URLs, call OpenAI, and write artifacts. | CLI run metadata and output directory |
| Diarization | Call `gpt-4o-transcribe-diarize` with `response_format="diarized_json"` and `chunking_strategy="auto"`. | Speaker-labeled segments |
| Transcript normalization | Convert API output into consistent JSON and Markdown. | `transcript_segments.json`, `speaker_labeled_transcript.md` |
| Meeting intelligence | Extract summary, decisions, actions, risks, questions, quotes, and follow-up email. | `meeting_intelligence.json`, `meeting_brief.md` |
| Guardrails and review gate | Redact sensitive fields, check evidence anchors, optionally moderate content, and route risky outputs for review. | `guardrail_report.json` |

This is intentionally request-based. The Realtime API is a better fit for live voice UX, browser capture, or telephony streaming. For durable post-call diarization, this pattern uses the Transcriptions API and then runs structured extraction over the speaker-labeled transcript.


## Why speaker-aware transcripts matter

The first version of meeting intelligence is often "send a transcript to a model and summarize it." That works for demos, but it breaks down in customer workflows because it loses who said what. A customer may state a requirement, a seller may make a commitment, and a manager may need the difference to be explicit.

Speaker-aware diarization gives the rest of the application better structure:

- Action items can include the speaker who committed to them.
- Risks can quote the exact customer concern.
- Follow-up email drafts can avoid attributing seller commitments to the customer.
- QA reviewers can spot-check speaker attribution by timestamp.
- CRM sync jobs can store evidence rather than opaque summaries.


## Security and guardrails

Treat meeting intelligence as a sensitive-data pipeline, not just a summarization job. Raw audio, speaker references, diarized transcripts, extracted notes, and downstream CRM records each need controls.

| Risk | Guardrail |
| --- | --- |
| Recording or speaker-reference misuse | Require consent and policy approval before recording, diarization, or reference-clip use. Treat speaker references as sensitive biometric-adjacent data. |
| Over-retention of raw audio | Do not save the raw transcription response by default. Keep raw audio and reference clips only as long as needed. Encrypt and restrict access if retained. |
| Prompt injection inside transcripts | Treat transcript text as untrusted evidence. Keep instructions in the system message and require the model to use only transcript-backed facts. |
| Unsupported action items or decisions | Use strict structured outputs and require timestamp anchors in evidence fields. |
| Sensitive content in generated notes | Run redaction before summarization where possible, then run post-generation checks on the transcript and brief. |
| Harmful or policy-sensitive content | Optionally call the Moderation API with `omni-moderation-latest` on transcript text and generated brief text. Moderation detects harmful content; it is not a replacement for privacy review. |
| Unsafe downstream writes | Do not write directly to CRM, ticketing, or analytics systems from the model output. Put a human review gate in front of medium/high risks, missing evidence, moderation flags, or raw-response retention. |
| Silent quality drift | Log model versions, prompt versions, schema versions, audio duration, redaction state, moderation state, and reviewer decisions. Sample calls for evals. |


## Prerequisites

- Python 3.10 or later.
- An OpenAI API key in `OPENAI_API_KEY` for real audio runs.
- A meeting recording in a supported audio format for real audio runs.
- Optional: up to four short, single-speaker reference clips. The speech-to-text guide recommends 2-10 second references, encoded as data URLs when sent with multipart form data.

The synthetic demo below uses only the Python standard library and does not call the API. For real audio, install dependencies from the example folder:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r examples/audio/speaker_aware_meeting_intelligence/requirements.txt
```


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

try:
    from IPython.display import JSON, Markdown, display
except ImportError:  # Makes this cell safe in non-notebook runners.
    JSON = None
    Markdown = None

    def display(value):
        print(value)


def find_example_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "examples/audio/speaker_aware_meeting_intelligence",
        Path("examples/audio/speaker_aware_meeting_intelligence"),
    ]
    for candidate in candidates:
        if (candidate / "meeting_intelligence.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Could not find meeting_intelligence.py. Run from the notebook folder or repo root.")


EXAMPLE_DIR = find_example_dir()
SCRIPT_PATH = EXAMPLE_DIR / "meeting_intelligence.py"
print(f"Example directory: {EXAMPLE_DIR}")

spec = importlib.util.spec_from_file_location("meeting_intelligence", SCRIPT_PATH)
meeting_intelligence = importlib.util.module_from_spec(spec)
assert spec and spec.loader
sys.modules["meeting_intelligence"] = meeting_intelligence
spec.loader.exec_module(meeting_intelligence)
print("Loaded meeting_intelligence.py")


## Step 1: Run the no-network demo

Start with the synthetic demo. It lets reviewers inspect the artifact contract and Markdown output without an API key.


In [ ]:
output_dir = Path(tempfile.mkdtemp(prefix="meeting-intelligence-demo-"))
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--demo",
    "--output-dir",
    str(output_dir),
]
result = subprocess.run(cmd, check=True, text=True, capture_output=True)
print(result.stdout)
print(f"Demo artifacts: {output_dir}")


In [ ]:
artifact_names = [
    "transcript_segments.json",
    "speaker_labeled_transcript.md",
    "meeting_intelligence.json",
    "meeting_brief.md",
    "guardrail_report.json",
]
for name in artifact_names:
    artifact_path = output_dir / name
    print(f"{name}: {artifact_path.exists()} ({artifact_path.stat().st_size} bytes)")


## Step 2: Inspect the speaker-labeled transcript

The pipeline normalizes diarized output into a small, stable segment schema. This becomes the contract between audio processing and meeting intelligence extraction.


In [ ]:
segments = json.loads((output_dir / "transcript_segments.json").read_text())
segments[:2]


In [ ]:
transcript_markdown = (output_dir / "speaker_labeled_transcript.md").read_text()
if Markdown:
    display(Markdown(transcript_markdown))
else:
    print(transcript_markdown)


## Step 3: Extract structured meeting intelligence

The sample sends the speaker-labeled transcript to a text model and asks for strict JSON with these fields:

- `summary`
- `participants`
- `customer_context`
- `decisions`
- `action_items`
- `risks`
- `open_questions`
- `notable_quotes`
- `follow_up_email`

The system instruction is deliberately conservative:

```text
Use only the transcript as evidence. Do not invent names, dates, decisions, or commitments.
If evidence is missing, leave the relevant array empty.
Include timestamps in every evidence field.
```

This matters because meeting intelligence often feeds systems of record. The safest default is to produce empty arrays instead of plausible but unsupported CRM notes.


In [ ]:
meeting_intelligence_json = json.loads((output_dir / "meeting_intelligence.json").read_text())
if JSON:
    display(JSON(meeting_intelligence_json, expanded=False))
else:
    print(json.dumps(meeting_intelligence_json, indent=2))


In [ ]:
brief_markdown = (output_dir / "meeting_brief.md").read_text()
if Markdown:
    display(Markdown(brief_markdown))
else:
    print(brief_markdown)


## Step 4: Review guardrail output

The sample writes `guardrail_report.json` with local checks for:

- normalized transcript segments;
- basic email and phone PII patterns;
- evidence fields without a timestamp anchor;
- medium/high risk outputs;
- optional moderation flags;
- raw transcription response storage.

Use `--fail-on-guardrail` in CI or batch jobs when review-required output should exit non-zero.


In [ ]:
guardrail_report = json.loads((output_dir / "guardrail_report.json").read_text())
if JSON:
    display(JSON(guardrail_report, expanded=True))
else:
    print(json.dumps(guardrail_report, indent=2))


## Step 5: Run with real audio

For a real recording, keep the first pass simple:

- Use one meeting audio file.
- Use `chunking_strategy="auto"` for longer recordings.
- Add known-speaker references only when you have consent and a clear business need.
- Use short, clean reference clips with one speaker and minimal background noise.

Known-speaker references are optional. Without them, diarization can still separate speakers, but labels may be generic, such as `speaker_0` or `speaker_1`. With references, the API can map segments to provided names.

The next cell is intentionally opt-in. Set `RUN_REAL_AUDIO = True`, provide your local audio paths, and make sure `OPENAI_API_KEY` is set.


In [ ]:
RUN_REAL_AUDIO = False
AUDIO_FILE = Path("/path/to/meeting.wav")
KNOWN_SPEAKERS = {
    # "Agent": Path("/path/to/agent_reference.wav"),
    # "Customer": Path("/path/to/customer_reference.wav"),
}
REAL_OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="meeting-intelligence-real-"))

if RUN_REAL_AUDIO:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY before running on real audio.")
    if not AUDIO_FILE.is_file():
        raise FileNotFoundError(AUDIO_FILE)

    real_cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        "--audio-file",
        str(AUDIO_FILE),
        "--redact",
        "--moderate",
        "--output-dir",
        str(REAL_OUTPUT_DIR),
    ]
    for speaker_name, reference_path in KNOWN_SPEAKERS.items():
        real_cmd.extend(["--known-speaker", f"{speaker_name}={reference_path}"])

    real_result = subprocess.run(real_cmd, check=True, text=True, capture_output=True)
    print(real_result.stdout)
else:
    print("Skipped real audio run. Set RUN_REAL_AUDIO = True after configuring AUDIO_FILE and OPENAI_API_KEY.")


### Core diarization request

The complete implementation lives in `meeting_intelligence.py`. The core API request is intentionally small:

```python
import base64
import mimetypes
from pathlib import Path

from openai import OpenAI


def to_data_url(path: Path) -> str:
    mime_type, _ = mimetypes.guess_type(path)
    if mime_type is None:
        mime_type = "audio/wav"
    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime_type};base64,{encoded}"


client = OpenAI()

with open("meeting.wav", "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-transcribe-diarize",
        file=audio_file,
        response_format="diarized_json",
        chunking_strategy="auto",
        extra_body={
            "known_speaker_names": ["Agent"],
            "known_speaker_references": [to_data_url(Path("agent_reference.wav"))],
        },
    )
```

The important details are:

- Use `response_format="diarized_json"` when you need segment-level speaker metadata.
- Use `chunking_strategy="auto"` for audio longer than 30 seconds.
- Pass known speaker names and references together, in the same order.
- Keep reference clips short and single-speaker.


## Production hardening checklist

Use this checklist before turning the sample into a customer workflow:

| Concern | Recommendation |
| --- | --- |
| Consent | Make sure call recording, diarization, and known-speaker references are permitted in your product, policy, and region. |
| Raw audio retention | Store raw audio only as long as needed. Persist normalized transcript segments when possible. |
| Speaker references | Treat reference clips as sensitive data. Store minimally, encrypt at rest, and rotate/delete when no longer needed. |
| Evidence | Require timestamped evidence on decisions, risks, and action items. |
| Human review | Route high-risk summaries, compliance promises, pricing claims, or contractual terms for review. |
| Moderation | Use the Moderation API for harmful-content classification when notes may contain unsafe content. Keep privacy and compliance checks separate. |
| Retry behavior | Retry transient API errors with backoff. Avoid duplicating downstream CRM writes by using idempotency keys. |
| Observability | Log model names, prompt versions, schema versions, audio duration, latency, redaction status, and reviewer decisions. |
| Evaluation | Sample calls weekly. Track speaker attribution accuracy, action-item precision, and unsupported-claim rate. |


## Optional: run the local test suite

The repository includes deterministic tests for the demo path, artifact creation, redaction, evidence anchors, raw-response retention, and the mocked Responses API call shape.


In [ ]:
RUN_LOCAL_TESTS = False

if RUN_LOCAL_TESTS:
    test_cmd = [sys.executable, str(EXAMPLE_DIR / "test_meeting_intelligence.py")]
    test_result = subprocess.run(test_cmd, check=True, text=True, capture_output=True)
    print(test_result.stdout)
else:
    print("Skipped tests. Set RUN_LOCAL_TESTS = True to run the deterministic local suite.")


## Next steps

You can adapt the same pipeline for:

- Customer success handoffs after quarterly business reviews.
- Support escalations where accountability and exact quotes matter.
- Sales discovery calls that feed CRM next steps.
- Recruiting interview debriefs where each interviewer needs sourced notes.
- Healthcare or financial-services workflows with stronger review and retention controls.

For live scenarios, use Realtime for the in-call experience and still run this post-call diarization pipeline when you need durable, evidence-backed meeting intelligence.

Useful docs:

- [Audio and speech guide](https://developers.openai.com/api/docs/guides/audio)
- [Speech-to-text and speaker diarization](https://developers.openai.com/api/docs/guides/speech-to-text)
- [Structured outputs](https://developers.openai.com/api/docs/guides/structured-outputs)
- [Moderation](https://developers.openai.com/api/docs/guides/moderation)
- [Safety best practices](https://developers.openai.com/api/docs/guides/safety-best-practices)
- [Realtime guide](https://developers.openai.com/api/docs/guides/realtime)
